# Notebook 04 — Engagement Prediction Model

**Project:** DReel AI v2  
**Algorithm:** Random Forest Regressor  
**Target:** User engagement rating (1–5)

Features are built from audio + vision pipelines. User ratings are stored in `data/engagement_ratings.csv`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from services.ml_service import (
    FEATURE_COLUMNS,
    build_feature_matrix,
    build_training_data,
    load_ratings,
    train_engagement_model,
)

## 1. Feature Matrix

In [2]:
features_df = build_feature_matrix()
ratings_df = load_ratings()

print(f"Reels with features: {len(features_df)}")
print(f"User ratings collected: {len(ratings_df)}")
features_df

Reels with features: 4
User ratings collected: 5


,reel_id,image_count,duration_sec,tempo_bpm,beat_count,avg_energy,avg_slide_duration,slide_std,avg_quality,min_quality,blur_mean,low_quality_ratio
0,6f542634-7b74-11f1-8cf5-04ecd80f8fe4,10,113.499,109.96,204,0.1739,11.350,1.776357e-15,0.500000,0.5000,0.000000,0.0
1,f08c2cc6-7b77-11f1-b783-04ecd80f8fe4,7,25.120,109.96,45,0.1876,3.589,5.120884e-01,0.500000,0.5000,0.000000,0.0
2,66432ef0-7b7a-11f1-afa6-04ecd80f8fe4,9,25.120,109.96,45,0.1876,2.791,4.235482e-01,0.500000,0.5000,0.000000,0.0
3,ab9bdc64-7b7c-11f1-bc17-04ecd80f8fe4,7,9.384,140.62,18,0.0653,1.341,4.330865e-01,0.873729,0.6828,1169.824286,0.0


In [3]:
x_data, y_data, real_mask = build_training_data()
training_df = x_data.copy()
training_df["engagement_score"] = y_data
training_df["is_real_rating"] = real_mask
training_df

,image_count,duration_sec,tempo_bpm,beat_count,avg_energy,avg_slide_duration,slide_std,avg_quality,min_quality,blur_mean,low_quality_ratio,engagement_score,is_real_rating
0,10,113.499,109.96,204,0.1739,11.350,1.776357e-15,0.500000,0.5000,0.000000,0.0,2.6,False
1,7,25.120,109.96,45,0.1876,3.589,5.120884e-01,0.500000,0.5000,0.000000,0.0,4.0,True
2,9,25.120,109.96,45,0.1876,2.791,4.235482e-01,0.500000,0.5000,0.000000,0.0,4.0,True
3,7,9.384,140.62,18,0.0653,1.341,4.330865e-01,0.873729,0.6828,1169.824286,0.0,5.0,True


## 2. Train Random Forest

In [4]:
metrics = train_engagement_model()
metrics

[ML] Engagement model trained: R²=0.8931, MAE=0.2263, real_ratings=3


{'trained': True,
 'samples': 4,
 'real_ratings': 3,
 'proxy_ratings': 1,
 'timestamp': '2026-07-20T06:46:29.124953+00:00',
 'cv_folds': 2,
 'cv_r2_mean': -2.8777,
 'cv_r2_std': 1.1137,
 'train_r2': 0.8931,
 'train_mae': 0.2263,
 'metrics_note': 'train_r2/train_mae are in-sample fit on the training data itself, not a held-out score, and will look overly optimistic. cv_r2_mean/cv_r2_std (k-fold cross-validation, when available) are the more honest estimate of generalization. Both are high-variance with this few samples — collect more real user ratings before quoting either as final performance.',
 'feature_importance': {'image_count': 0.1125,
  'duration_sec': 0.1311,
  'tempo_bpm': 0.0238,
  'beat_count': 0.1333,
  'avg_energy': 0.0364,
  'avg_slide_duration': 0.0968,
  'slide_std': 0.3472,
  'avg_quality': 0.0597,
  'min_quality': 0.036,
  'blur_mean': 0.0232,
  'low_quality_ratio': 0.0}}

In [5]:
model = RandomForestRegressor(n_estimators=120, max_depth=5, random_state=42)
model.fit(x_data, y_data)
predictions = model.predict(x_data)

print(f"Train R²:   {r2_score(y_data, predictions):.4f}")
print(f"Train MAE:  {mean_absolute_error(y_data, predictions):.4f}")

Train R²:   0.8931
Train MAE:  0.2263


## 3. Feature Importance

In [6]:
importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["feature"], importance_df["importance"], color="#2575fc")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

C:\Users\manuc\AppData\Local\Temp\ipykernel_4480\4079550375.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Actual vs Predicted

In [7]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_data, y=predictions, s=100, color="#6a11cb")
plt.plot([1, 5], [1, 5], "--", color="#ff4d4d", label="Perfect prediction")
plt.xlabel("Actual Engagement Score")
plt.ylabel("Predicted Engagement Score")
plt.title("Model Performance")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\manuc\AppData\Local\Temp\ipykernel_4480\712887209.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Conclusions

- The model uses **11 features** from audio beat-sync and computer vision pipelines.
- **User ratings** in the gallery improve the model; unrated reels use proxy labels until rated.
- Top features typically include `avg_quality`, `avg_slide_duration`, and `tempo_bpm`.
- Collect more ratings (20+) for stronger cross-validation and deployment confidence.